In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

train = pd.read_csv('../data/raw/train.csv')
print(f"Shape: {train.shape}")

In [ ]:
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
 
y = np.log1p(train['SalePrice'])
X = train.drop(['SalePrice', 'Id'], axis=1)

print(f"X: {X.shape}, y: {y.shape}")

In [ ]:
X['TotalSF'] = X['1stFlrSF'] + X['2ndFlrSF'] + X['TotalBsmtSF']

In [ ]:
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

cat_cols = X.select_dtypes(include=['object']).columns
X[cat_cols] = X[cat_cols].fillna('Missing')

X = pd.get_dummies(X, drop_first=True)

print(f"After preprocessing: {X.shape}")
print(f"NaN count: {X.isnull().sum().sum()}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt_scores = -cross_val_score(dt, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Decision Tree:  {dt_scores.mean():.4f}, std: {dt_scores.std():.4f}")

# Random Forest
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_scores = -cross_val_score(rf, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Random Forest:  {rf_scores.mean():.4f}, std: {rf_scores.std():.4f}")

# XGBoost
xgb = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
xgb_scores = -cross_val_score(xgb, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"XGBoost:        {xgb_scores.mean():.4f}, std: {xgb_scores.std():.4f}")

# LightGBM
lgbm = LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1)
lgbm_scores = -cross_val_score(lgbm, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"LightGBM:       {lgbm_scores.mean():.4f}, std: {lgbm_scores.std():.4f}")

# Ridge (with scaling) - baseline reference
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge(alpha=1.0)
ridge_scores = -cross_val_score(ridge, X_scaled, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Ridge:          {ridge_scores.mean():.4f} ± {ridge_scores.std():.4f}")

## Model selection

LightGBM beats previous floor (Ridge 0.1289) without any tuning.

| Model          | RMSE   | Std     |
|----------------|--------|---------|
| Decision Tree  | 0.2088 | 0.0111  |
| Random Forest  | 0.1372 | 0.0078  |
| XGBoost        | 0.1369 | 0.0075  |
| **LightGBM**       | **0.1282** | **0.0055**  |
| Ridge          | 0.1289 | 0.0129  |


We have slightly lower RMSE (0.1282) with LightGBM model.

Also the LightGBM model has the lowest standard deviation (0.0055)

Our next steps involve hyperparameter tuning with Optuna

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'verbosity': -1,
        'n_jobs': -1,
    }
    model = LGBMRegressor(**params)
    scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_root_mean_squared_error")

    return scores.mean()

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar = True)

print(f"\nBest RMSE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbosity'] = -1
best_params['n_jobs'] = -1

best_params

In [ ]:
final_model = LGBMRegressor(**best_params)
final_scores = -cross_val_score(final_model, X, y, cv=kf, scoring='neg_root_mean_squared_error')

print(f"Final LightGBM RMSE: {final_scores.mean():.4f}, std: {final_scores.std():.4f}")

## Optuna tuning

Used Optuna to tune a LightGBM model by trying different hyperparameter combinations automatically.

Then I evaluated each model with cross validation using RMSE as the metric.

Model with Optuna tuning performed better (smaller loss, 0.1190 RMSE) than the previous LightGBM model (loss 0.1282) without tuning.

Hyperparameter tuning significantly improved the model performance and helped to find more optimized LightGBM config

**Best params (Optuna):**
- n_estimators: 554
- learning_rate: 0.024
- max_depth: 3
- num_leaves: 117